# F1 Adverse Conditions Project -- SQL Queries
## BUSN 32120 | Winter 2026

This notebook contains all required SQL queries for the final project. Queries
are executed against an in-memory SQLite database built from the same API data
used in the main analysis notebook. Each query is commented with what it does,
why it is relevant to the analysis, and a summary of its output.

**Tables in the database:**
- `results` -- one row per driver per race (Dataset 1a)
- `qualifying` -- one row per driver per qualifying session (Dataset 1b)
- `weather` -- one row per race round with conditions and circuit metadata (Dataset 2)


## 1. Setup and Data Loading

In [ ]:
import requests, time, sqlite3
import pandas as pd

JOLPICA = "https://api.jolpi.ca/ergast/f1"
OPENF1  = "https://api.openf1.org/v1"
SEASONS = [2022, 2023, 2024, 2025]

CTOR_MAP = {
    "Red Bull": "Red Bull",        "McLaren": "McLaren",
    "Ferrari": "Ferrari",          "Mercedes": "Mercedes",
    "Alpine": "Alpine",            "Alpine F1 Team": "Alpine",
    "Williams": "Williams",        "Aston Martin": "Aston Martin",
    "Haas F1 Team": "Haas",        "Haas": "Haas",
    "Kick Sauber": "Sauber",       "Sauber": "Sauber",
    "Alfa Romeo": "Sauber",        "RB": "Racing Bulls",
    "Racing Bulls": "Racing Bulls", "AlphaTauri": "Racing Bulls",
}
CTOR_TIER = {
    "Red Bull": "Top",    "McLaren": "Top",
    "Ferrari": "Top",     "Mercedes": "Top",
    "Williams": "Mid",    "Alpine": "Mid",
    "Aston Martin": "Mid","Racing Bulls": "Mid",
    "Haas": "Back",       "Sauber": "Back",
}

CIRCUIT_META = {
    "albert_park":("Permanent",10,0),"bahrain":("Permanent",14,0),
    "jeddah":("Hybrid",15,1),"suzuka":("Permanent",45,0),
    "shanghai":("Permanent",7,0),"miami":("Permanent",4,0),
    "imola":("Permanent",31,0),"monaco":("Street",32,1),
    "villeneuve":("Permanent",9,0),"catalunya":("Permanent",115,0),
    "red_bull_ring":("Permanent",693,0),"silverstone":("Permanent",126,0),
    "hungaroring":("Permanent",240,0),"spa":("Permanent",400,0),
    "zandvoort":("Permanent",9,0),"monza":("Permanent",162,0),
    "baku":("Street",0,1),"marina_bay":("Street",5,1),
    "austin":("Permanent",169,0),"rodriguez":("Permanent",2240,0),
    "interlagos":("Permanent",799,0),"las_vegas":("Street",636,1),
    "losail":("Permanent",32,0),"yas_marina":("Permanent",3,0),
    "ricard":("Permanent",103,0),"portimao":("Permanent",108,0),
    "mugello":("Permanent",275,0),"nurburgring":("Permanent",578,0),
    "istanbul":("Permanent",130,0),
}
KNOWN_WET = {
    (2022,"suzuka"),(2022,"interlagos"),(2022,"marina_bay"),
    (2023,"monaco"),(2023,"villeneuve"),
    (2024,"spa"),(2024,"interlagos"),(2025,"shanghai"),
}
KNOWN_MIXED = {
    (2022,"imola"),(2022,"albert_park"),
    (2023,"albert_park"),(2023,"silverstone"),
    (2024,"suzuka"),(2024,"silverstone"),(2025,"bahrain"),
}

CID_TO_SHORT = {
    "albert_park":"melbourne","bahrain":"sakhir","jeddah":"jeddah",
    "suzuka":"suzuka","shanghai":"shanghai","miami":"miami",
    "imola":"imola","monaco":"monaco","villeneuve":"montreal",
    "catalunya":"barcelona","red_bull_ring":"spielberg","silverstone":"silverstone",
    "hungaroring":"budapest","spa":"spa","zandvoort":"zandvoort",
    "monza":"monza","baku":"baku","marina_bay":"singapore",
    "austin":"austin","rodriguez":"mexico city","interlagos":"sao paulo",
    "las_vegas":"las vegas","losail":"lusail","yas_marina":"abu dhabi",
    "ricard":"le castellet","portimao":"portimao","mugello":"mugello",
    "nurburgring":"nurburgring","istanbul":"istanbul",
}

def fetch(url, retries=3):
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"  [{attempt+1}/{retries}] {e}")
            time.sleep(1.0 * (attempt + 1))
    return None


In [ ]:
# ── Pull results + qualifying from Jolpica ────────────────────────────────────
result_rows, qual_rows = [], []

for season in SEASONS:
    print(f"Pulling {season} results ...")
    data = fetch(f"{JOLPICA}/{season}/results/?format=json&limit=500")
    for race in (data["MRData"]["RaceTable"]["Races"] if data else []):
        rnd = int(race["round"])
        cid = race["Circuit"]["circuitId"]
        for r in race["Results"]:
            pos  = int(r["position"]) if r["position"].isdigit() else None
            grid = int(r.get("grid", 0)) or None
            result_rows.append({
                "year": season, "round": rnd,
                "race_name": race["raceName"],
                "circuit_id": cid,
                "circuit_name": race["Circuit"]["circuitName"],
                "country": race["Circuit"]["Location"]["country"],
                "driver_id": r["Driver"]["driverId"],
                "driver_name": r["Driver"]["givenName"] + " " + r["Driver"]["familyName"],
                "constructor": CTOR_MAP.get(r["Constructor"]["name"], r["Constructor"]["name"]),
                "grid": grid, "position": pos,
                "laps": int(r.get("laps", 0)),
                "status": r.get("status", "Unknown"),
                "fastest_lap": int(r.get("FastestLap", {}).get("rank") == "1"),
                "points": float(r.get("points", 0)),
            })
    time.sleep(0.4)

    print(f"Pulling {season} qualifying ...")
    data = fetch(f"{JOLPICA}/{season}/qualifying/?format=json&limit=500")
    for race in (data["MRData"]["RaceTable"]["Races"] if data else []):
        rnd = int(race["round"])
        for q in race.get("QualifyingResults", []):
            qual_rows.append({
                "year": season, "round": rnd,
                "driver_id": q["Driver"]["driverId"],
                "qual_pos": int(q["position"]),
            })
    time.sleep(0.4)

df_res  = pd.DataFrame(result_rows)
df_qual = pd.DataFrame(qual_rows)
df_res["ctor_tier"] = df_res["constructor"].map(CTOR_TIER).fillna("Mid")
print(f"\nResults: {len(df_res)} rows | Qualifying: {len(df_qual)} rows")


In [ ]:
# ── Pull weather from OpenF1 ───────────────────────────────────────────────────
session_map = {}
for season in SEASONS:
    data = fetch(f"{OPENF1}/sessions?year={season}&session_type=Race")
    if data:
        for s in data:
            key = (season, str(s.get("circuit_short_name", "")).lower())
            session_map[key] = s.get("session_key")
    time.sleep(0.4)

weather_rows = []
race_rounds = df_res.drop_duplicates(["year","round"])[["year","round","circuit_id"]].copy()

for _, row in race_rounds.iterrows():
    yr, rnd, cid = row["year"], row["round"], row["circuit_id"]
    short = CID_TO_SHORT.get(cid, "")
    sk = session_map.get((yr, short)) or session_map.get((yr, short.split()[0]))

    rain_pct, track_temp, air_temp, humidity, condition = None, None, None, None, None
    if sk:
        data = fetch(f"{OPENF1}/weather?session_key={sk}")
        if data and len(data) > 0:
            wdf = pd.DataFrame(data)
            if "rainfall" in wdf.columns:
                rain_pct  = round(float(wdf["rainfall"].mean()), 3)
                condition = ("Wet" if rain_pct > 0.30 else
                             "Mixed" if rain_pct > 0.05 else "Dry")
            if "track_temperature" in wdf.columns:
                track_temp = round(float(wdf["track_temperature"].mean()), 1)
            if "air_temperature" in wdf.columns:
                air_temp = round(float(wdf["air_temperature"].mean()), 1)
            if "humidity" in wdf.columns:
                humidity = round(float(wdf["humidity"].mean()), 1)
        time.sleep(0.2)

    if condition is None:
        if (yr, cid) in KNOWN_WET:     condition = "Wet"
        elif (yr, cid) in KNOWN_MIXED: condition = "Mixed"
        else:                          condition = "Dry"

    meta = CIRCUIT_META.get(cid, ("Permanent", 50, 0))
    weather_rows.append({
        "year": yr, "round": rnd, "circuit_id": cid,
        "condition": condition, "rain_pct": rain_pct,
        "track_temp": track_temp, "air_temp": air_temp,
        "humidity": humidity, "track_type": meta[0],
        "altitude_m": meta[1], "is_street": meta[2],
        "high_altitude": int(meta[1] >= 500),
    })

df_weather = pd.DataFrame(weather_rows)
print(f"Weather: {len(df_weather)} race-rounds")
print(df_weather["condition"].value_counts().to_string())


## 2. Load Data into SQLite

In [ ]:
# ── Create in-memory SQLite database and load all three tables ─────────────────
conn = sqlite3.connect(":memory:")

df_res.to_sql("results",    conn, index=False, if_exists="replace")
df_qual.to_sql("qualifying", conn, index=False, if_exists="replace")
df_weather.to_sql("weather", conn, index=False, if_exists="replace")

# Quick verification: show table schemas and row counts
for table in ["results", "qualifying", "weather"]:
    schema = pd.read_sql(f"PRAGMA table_info({table})", conn)
    count  = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn).iloc[0, 0]
    print(f"\n{table.upper()} ({count} rows):")
    print(schema[["name", "type"]].to_string(index=False))

# Helper function to run a query and display results
def run(sql, n=None):
    """Execute SQL and return a DataFrame. If n is set, only show n rows."""
    result = pd.read_sql(sql, conn)
    if n:
        print(result.head(n).to_string(index=False))
    else:
        print(result.to_string(index=False))
    return result


## 3. SQL Queries

### Query 1 — Constructor standings by season (GROUP BY)

**What:** Aggregates total points, wins, and podiums per constructor per season.

**Why:** Establishes the competitive hierarchy each year. Knowing which teams were
dominant in which seasons is essential context for all downstream analysis -- if
Red Bull won 80% of races in 2023, a driver's results that year must be interpreted
in that light.

**Technique:** GROUP BY with multiple aggregation functions (SUM, COUNT with CASE).


In [ ]:
# QUERY 1: Constructor standings by season
# GROUP BY constructor and year to see which teams dominated each season.
# Uses CASE WHEN inside SUM to count wins (position = 1) and podiums (position <= 3).
# Ordered by year then total points descending to show the standings.

q1 = """
SELECT  r.year,
        r.constructor,
        r.ctor_tier,
        SUM(r.points)                                   AS total_points,
        SUM(CASE WHEN r.position = 1 THEN 1 ELSE 0 END) AS wins,
        SUM(CASE WHEN r.position <= 3 THEN 1 ELSE 0 END) AS podiums,
        COUNT(*)                                         AS entries,
        ROUND(AVG(r.position), 1)                        AS avg_finish
FROM    results r
WHERE   r.position IS NOT NULL
GROUP BY r.year, r.constructor, r.ctor_tier
ORDER BY r.year, total_points DESC
"""

print("QUERY 1: Constructor Standings by Season")
print("Shows the points pecking order each year — essential context for interpreting driver results.\n")
_ = run(q1)


### Query 2 — DNF rates by weather condition (GROUP BY)

**What:** Groups all race entries by weather condition (Dry / Mixed / Wet) and
calculates the DNF rate, average finish, and total entries for each.

**Why:** Central to the project thesis. If wet races have materially higher DNF
rates, it confirms that adverse conditions introduce chaos and attrition -- the
mechanism by which driver skill might matter more.

**Technique:** GROUP BY with JOIN to weather table, plus CASE WHEN for DNF classification.


In [ ]:
# QUERY 2: DNF rates by weather condition
# JOINs results with weather to get the condition for each race entry.
# Classifies DNF using the same logic as the main notebook: if status is not
# 'Finished' and doesn't start with '+' or 'Lapped', it's a DNF.
# Groups by condition and computes the DNF rate.

q2 = """
SELECT  w.condition,
        COUNT(*)                                                     AS total_entries,
        SUM(CASE WHEN r.status NOT IN ('Finished')
                  AND r.status NOT LIKE '+%'
                  AND r.status NOT LIKE 'Lapped%'
             THEN 1 ELSE 0 END)                                      AS dnfs,
        ROUND(
          SUM(CASE WHEN r.status NOT IN ('Finished')
                    AND r.status NOT LIKE '+%'
                    AND r.status NOT LIKE 'Lapped%'
               THEN 1.0 ELSE 0.0 END) / COUNT(*), 3)                AS dnf_rate,
        ROUND(AVG(r.position), 1)                                    AS avg_finish
FROM    results r
        JOIN weather w ON r.year = w.year AND r.round = w.round
GROUP BY w.condition
ORDER BY dnf_rate DESC
"""

print("QUERY 2: DNF Rate by Weather Condition")
print("Tests whether wet/mixed conditions produce more retirements than dry races.\n")
_ = run(q2)


### Query 3 — Qualifying vs race finish: who gains the most? (JOIN)

**What:** Joins results with qualifying to compute positions gained (qual_pos minus
finish position) for every driver entry, then aggregates by driver.

**Why:** Drivers who consistently gain places from qualifying to the finish are
either great racers, great strategists, or both. This is a key signal of race-day
skill separate from one-lap qualifying pace.

**Technique:** INNER JOIN results with qualifying on year + round + driver_id.


In [ ]:
# QUERY 3: Qualifying to finish — biggest position gainers
# JOINs results with qualifying on the composite key (year, round, driver_id).
# Computes positions gained as (qual_pos - finish position); positive = gained places.
# Filters to drivers with at least 20 race starts for sample size.

q3 = """
SELECT  r.driver_name,
        r.constructor,
        COUNT(*)                                        AS races,
        ROUND(AVG(q.qual_pos), 1)                       AS avg_qual,
        ROUND(AVG(r.position), 1)                       AS avg_finish,
        ROUND(AVG(q.qual_pos - r.position), 2)          AS avg_places_gained,
        ROUND(AVG(r.points), 2)                         AS avg_points
FROM    results r
        INNER JOIN qualifying q
            ON r.year = q.year
            AND r.round = q.round
            AND r.driver_id = q.driver_id
WHERE   r.position IS NOT NULL
GROUP BY r.driver_name, r.constructor
HAVING  COUNT(*) >= 20
ORDER BY avg_places_gained DESC
"""

print("QUERY 3: Biggest Qual-to-Finish Gainers (min 20 races)")
print("Who gains the most positions on race day? Positive = consistently gaining places.\n")
_ = run(q3)


### Query 4 — Wet-race specialists: three-way JOIN (JOIN)

**What:** Joins results, qualifying, and weather to isolate driver performance
specifically in wet and mixed conditions, comparing their wet avg finish to their
overall avg finish.

**Why:** This is the core question of the project: which drivers outperform their
baseline when conditions are adverse? The three-way join is necessary because
weather data lives in a separate dataset from results and qualifying.

**Technique:** Three-table JOIN (results + qualifying + weather) with conditional
aggregation.


In [ ]:
# QUERY 4: Wet-race specialists via three-way JOIN
# Joins all three tables: results for outcomes, qualifying for grid context,
# and weather for the race condition. Filters to Wet or Mixed conditions.
# Compares each driver's adverse-condition avg finish to their overall avg
# (computed inline). Only shows drivers with 3+ adverse starts.

q4 = """
SELECT  r.driver_name,
        r.constructor,
        r.ctor_tier,
        COUNT(*)                              AS adverse_races,
        ROUND(AVG(r.position), 1)             AS avg_finish_adverse,
        ROUND(AVG(q.qual_pos), 1)             AS avg_qual_adverse,
        ROUND(AVG(q.qual_pos - r.position), 2) AS avg_gained_adverse
FROM    results r
        INNER JOIN qualifying q
            ON  r.year = q.year
            AND r.round = q.round
            AND r.driver_id = q.driver_id
        INNER JOIN weather w
            ON  r.year = w.year
            AND r.round = w.round
WHERE   r.position IS NOT NULL
        AND w.condition IN ('Wet', 'Mixed')
GROUP BY r.driver_name, r.constructor, r.ctor_tier
HAVING  COUNT(*) >= 3
ORDER BY avg_finish_adverse ASC
"""

print("QUERY 4: Wet/Mixed Race Performance (min 3 starts, three-way JOIN)")
print("Best average finishers in adverse conditions — core question of the project.\n")
_ = run(q4)


### Query 5 — Street circuit performance by constructor tier (JOIN)

**What:** Joins results with weather (which carries the is_street flag from
Dataset 2) to compare constructor performance on street vs permanent circuits.

**Why:** Street circuits are thought to be "great equalisers" -- tight walls and
fewer overtaking opportunities could compress the field. This query tests that
hypothesis by tier.

**Technique:** JOIN results with weather, GROUP BY tier and street flag.


In [ ]:
# QUERY 5: Street circuit performance by constructor tier
# JOINs results with weather to access the is_street flag (from Dataset 2).
# Groups by constructor tier and street/permanent to see if the gap between
# top and back teams narrows on street circuits.

q5 = """
SELECT  r.ctor_tier,
        CASE WHEN w.is_street = 1 THEN 'Street' ELSE 'Permanent' END AS circuit_type,
        COUNT(*)                                                       AS entries,
        ROUND(AVG(r.position), 2)                                      AS avg_finish,
        ROUND(AVG(r.points), 2)                                        AS avg_points,
        SUM(CASE WHEN r.position <= 3 THEN 1 ELSE 0 END)              AS podiums
FROM    results r
        JOIN weather w ON r.year = w.year AND r.round = w.round
WHERE   r.position IS NOT NULL
GROUP BY r.ctor_tier,
         CASE WHEN w.is_street = 1 THEN 'Street' ELSE 'Permanent' END
ORDER BY r.ctor_tier, circuit_type
"""

print("QUERY 5: Constructor Tier Performance on Street vs Permanent Circuits")
print("Tests whether street circuits compress the gap between top and back teams.\n")
_ = run(q5)


### Query 6 — Cumulative points race: running championship totals (WINDOW FUNCTION)

**What:** Uses a running SUM window function to show each driver's cumulative points
total as the season progresses, round by round.

**Why:** The championship is decided by cumulative points. Visualising the running
total reveals momentum shifts -- when did Verstappen pull away? When did McLaren
close the gap? This is the kind of output a data journalist would want.

**Technique:** SUM() OVER (PARTITION BY ... ORDER BY ...) window function.


In [ ]:
# QUERY 6: Cumulative championship points using a WINDOW FUNCTION
# PARTITION BY year, driver_name so each driver-season gets its own running total.
# ORDER BY round so the cumulative sum builds chronologically.
# Filtered to 2024 and top 6 point scorers for readability.

q6 = """
WITH season_points AS (
    SELECT  year,
            round,
            driver_name,
            points,
            SUM(points) OVER (
                PARTITION BY year, driver_name
                ORDER BY round
            ) AS cumulative_points
    FROM    results
    WHERE   year = 2024
)
SELECT  sp.round,
        sp.driver_name,
        sp.points         AS round_points,
        sp.cumulative_points
FROM    season_points sp
WHERE   sp.driver_name IN (
            SELECT  driver_name
            FROM    results
            WHERE   year = 2024
            GROUP BY driver_name
            ORDER BY SUM(points) DESC
            LIMIT 6
        )
ORDER BY sp.driver_name, sp.round
"""

print("QUERY 6: 2024 Cumulative Points Race (top 6 drivers, WINDOW FUNCTION)")
print("Running SUM partitioned by driver shows the championship battle unfolding.\n")
_ = run(q6, n=30)
print("\n... (showing first 30 rows)")


### Query 7 — Positions gained rank within each race (WINDOW FUNCTION)

**What:** For each race, ranks all drivers by how many positions they gained from
grid to finish using RANK() OVER.

**Why:** Identifies the "driver of the day" type performances -- who moved through
the field most effectively in each race? Repeated appearances at the top of this
ranking signal genuine race craft.

**Technique:** RANK() OVER (PARTITION BY year, round ORDER BY ...) window function.


In [ ]:
# QUERY 7: Rank drivers within each race by positions gained (WINDOW FUNCTION)
# Uses RANK() OVER to assign a within-race rank based on grid - position (places gained).
# Filters to the top-3 gainers per race to keep output manageable.
# This highlights the standout drives where drivers cut through the field.

q7 = """
WITH race_gains AS (
    SELECT  year,
            round,
            race_name,
            driver_name,
            constructor,
            grid,
            position,
            (grid - position)  AS places_gained,
            RANK() OVER (
                PARTITION BY year, round
                ORDER BY (grid - position) DESC
            ) AS gain_rank
    FROM    results
    WHERE   position IS NOT NULL
            AND grid IS NOT NULL
            AND grid > 0
)
SELECT  year, round, race_name, driver_name, constructor,
        grid, position, places_gained, gain_rank
FROM    race_gains
WHERE   gain_rank <= 3
ORDER BY year, round, gain_rank
"""

print("QUERY 7: Top 3 Position Gainers Per Race (RANK WINDOW FUNCTION)")
print("Highlights the standout drives each weekend — who cut through the field?\n")
_ = run(q7, n=30)
print("\n... (showing first 30 rows)")


### Query 8 — Drivers who finish better in wet than their own average (SUBQUERY)

**What:** Uses a correlated subquery to find drivers whose average finishing
position in wet/mixed races is better (lower) than their overall average.

**Why:** Directly identifies the wet-weather specialists. By comparing each driver
to their own baseline (not to the field), we control for car quality -- a Sauber
driver who finishes P12 in the wet vs P16 overall is arguably more impressive
than a Red Bull driver finishing P3 in the wet vs P2 overall.

**Technique:** Correlated subquery in HAVING clause that references the outer query.


In [ ]:
# QUERY 8: Wet-weather outperformers via correlated SUBQUERY
# The outer query groups results in Wet/Mixed conditions by driver.
# The HAVING clause contains a correlated subquery that computes each driver's
# overall average finish — only drivers whose wet avg is BETTER (lower) are kept.
# This is a correlated subquery because the inner query references the outer
# query's driver_name.

q8 = """
SELECT  r.driver_name,
        r.constructor,
        COUNT(*)                  AS wet_races,
        ROUND(AVG(r.position), 2) AS avg_wet_finish,
        ROUND((
            SELECT AVG(r2.position)
            FROM   results r2
            WHERE  r2.driver_name = r.driver_name
               AND r2.position IS NOT NULL
        ), 2)                     AS avg_overall_finish,
        ROUND(
            (SELECT AVG(r2.position)
             FROM   results r2
             WHERE  r2.driver_name = r.driver_name
               AND  r2.position IS NOT NULL)
            - AVG(r.position), 2
        )                         AS wet_advantage
FROM    results r
        JOIN weather w ON r.year = w.year AND r.round = w.round
WHERE   r.position IS NOT NULL
        AND w.condition IN ('Wet', 'Mixed')
GROUP BY r.driver_name, r.constructor
HAVING  AVG(r.position) < (
            SELECT AVG(r2.position)
            FROM   results r2
            WHERE  r2.driver_name = r.driver_name
               AND r2.position IS NOT NULL
        )
        AND COUNT(*) >= 3
ORDER BY wet_advantage DESC
"""

print("QUERY 8: Wet-Weather Outperformers (correlated SUBQUERY)")
print("Drivers whose wet avg finish is BETTER than their own overall avg (min 3 wet races).")
print("wet_advantage = how many positions better they do in the wet.\n")
_ = run(q8)


### Query 9 — High-DNF circuits: above-average attrition tracks (SUBQUERY)

**What:** Uses a scalar subquery to compute the overall DNF rate, then filters to
circuits whose DNF rate exceeds that average.

**Why:** Feeds directly into the "dangerous circuits" analysis in the main notebook.
Circuits with high attrition are where reliability and composure become performance
differentiators.

**Technique:** Scalar subquery in WHERE clause to compute a dynamic threshold.


In [ ]:
# QUERY 9: Circuits with above-average DNF rates (scalar SUBQUERY)
# The subquery computes the overall DNF rate across all entries.
# The outer query groups by circuit and keeps only those whose DNF rate
# exceeds the global average. This dynamically adapts the threshold
# rather than hardcoding it.

q9 = """
SELECT  r.circuit_name,
        r.circuit_id,
        COUNT(*)                                                     AS entries,
        SUM(CASE WHEN r.status NOT IN ('Finished')
                  AND r.status NOT LIKE '+%'
                  AND r.status NOT LIKE 'Lapped%'
             THEN 1 ELSE 0 END)                                      AS dnfs,
        ROUND(
          SUM(CASE WHEN r.status NOT IN ('Finished')
                    AND r.status NOT LIKE '+%'
                    AND r.status NOT LIKE 'Lapped%'
               THEN 1.0 ELSE 0.0 END) / COUNT(*), 3)                AS dnf_rate
FROM    results r
GROUP BY r.circuit_name, r.circuit_id
HAVING  dnf_rate > (
            SELECT  SUM(CASE WHEN status NOT IN ('Finished')
                              AND status NOT LIKE '+%'
                              AND status NOT LIKE 'Lapped%'
                         THEN 1.0 ELSE 0.0 END) / COUNT(*)
            FROM    results
        )
ORDER BY dnf_rate DESC
"""

print("QUERY 9: Circuits with Above-Average DNF Rates (scalar SUBQUERY)")
print("Dynamically computes the global DNF rate and filters to circuits that exceed it.\n")
_ = run(q9)


### Query 10 — Teammate head-to-head battles by season (GROUP BY + SELF-JOIN)

**What:** For each constructor-season pair where two drivers shared the car,
computes how often each beat the other when both finished.

**Why:** The most direct measure of driver skill in F1. If two drivers have the
same car and one consistently finishes ahead, the difference is the driver, not the
machinery. This feeds the "car vs driver" decomposition in the main analysis.

**Technique:** Self-join on results (same year, round, constructor), GROUP BY,
HAVING to filter to meaningful sample sizes.


In [ ]:
# QUERY 10: Teammate head-to-head via SELF-JOIN
# Self-joins results to itself: for each race where two drivers from the same
# constructor both finished, determines who finished ahead.
# Uses r1.driver_id < r2.driver_id to avoid counting each pair twice.
# Groups by season and constructor to show the within-team battle.

q10 = """
SELECT  r1.year,
        r1.constructor,
        r1.driver_name                                       AS driver_1,
        r2.driver_name                                       AS driver_2,
        COUNT(*)                                              AS races_both_finished,
        SUM(CASE WHEN r1.position < r2.position THEN 1 ELSE 0 END) AS d1_wins,
        SUM(CASE WHEN r2.position < r1.position THEN 1 ELSE 0 END) AS d2_wins
FROM    results r1
        INNER JOIN results r2
            ON  r1.year = r2.year
            AND r1.round = r2.round
            AND r1.constructor = r2.constructor
            AND r1.driver_id < r2.driver_id
WHERE   r1.position IS NOT NULL
        AND r2.position IS NOT NULL
GROUP BY r1.year, r1.constructor, r1.driver_name, r2.driver_name
HAVING  COUNT(*) >= 5
ORDER BY r1.year, r1.constructor
"""

print("QUERY 10: Teammate Head-to-Head Battles (SELF-JOIN)")
print("The purest driver skill signal — same car, different results.\n")
_ = run(q10)


### Query 11 — Points distribution: how concentrated is success? (GROUP BY)

**What:** Groups drivers into points tiers and counts how many fall into each
bucket, revealing how concentrated or dispersed championship points are.

**Why:** Answers the question: is F1 a sport where a handful of drivers collect
all the points, or is it more evenly spread? This has direct implications for
whether the "car vs driver" debate even matters -- if only 5 drivers ever score
meaningful points, the car is doing most of the talking.

**Technique:** GROUP BY with CASE WHEN to create custom buckets.


In [ ]:
# QUERY 11: Points distribution buckets
# Groups drivers by total career points (2022-2025) into tiers using CASE WHEN.
# This shows how concentrated success is — essential context for the target audience.

q11 = """
SELECT  points_tier,
        COUNT(*)             AS num_drivers,
        ROUND(AVG(total), 1) AS avg_points_in_tier,
        MIN(total)           AS min_pts,
        MAX(total)           AS max_pts
FROM (
    SELECT  driver_name,
            SUM(points) AS total,
            CASE
                WHEN SUM(points) >= 500 THEN '1. Elite (500+)'
                WHEN SUM(points) >= 200 THEN '2. Contender (200-499)'
                WHEN SUM(points) >= 50  THEN '3. Points regular (50-199)'
                WHEN SUM(points) > 0    THEN '4. Occasional scorer (1-49)'
                ELSE                         '5. Zero points'
            END AS points_tier
    FROM    results
    GROUP BY driver_name
)
GROUP BY points_tier
ORDER BY points_tier
"""

print("QUERY 11: Points Distribution by Driver Tier (GROUP BY + CASE)")
print("How concentrated is F1 success? Are points spread evenly or hoarded by a few?\n")
_ = run(q11)


### Query 12 — Rolling 5-race average finish by driver (WINDOW FUNCTION + JOIN)

**What:** Computes a rolling 5-race moving average of finishing position for each
driver, joined with weather conditions to flag which stretches included adverse
conditions.

**Why:** A rolling average smooths noise and reveals form trends. If a driver's
rolling average dips during a stretch of wet races, that is evidence of condition
sensitivity. Conversely, if it stays flat or improves, they are a genuine
all-conditions performer.

**Technique:** AVG() OVER (PARTITION BY ... ORDER BY ... ROWS BETWEEN ...) window
function combined with a JOIN to the weather table.


In [ ]:
# QUERY 12: Rolling 5-race average finish with weather context (WINDOW + JOIN)
# Uses AVG() OVER with ROWS BETWEEN to compute a trailing 5-race moving average
# of finishing position for each driver. Joined with weather so we can see
# whether dips in form coincide with adverse conditions.
# Filtered to Verstappen and Norris in 2024 for a focused comparison.

q12 = """
SELECT  r.driver_name,
        r.round,
        r.race_name,
        r.position,
        w.condition,
        ROUND(
            AVG(r.position) OVER (
                PARTITION BY r.driver_name
                ORDER BY r.round
                ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
            ), 2
        ) AS rolling_5_avg
FROM    results r
        JOIN weather w ON r.year = w.year AND r.round = w.round
WHERE   r.year = 2024
        AND r.position IS NOT NULL
        AND r.driver_name IN ('Max Verstappen', 'Lando Norris')
ORDER BY r.driver_name, r.round
"""

print("QUERY 12: Rolling 5-Race Avg Finish — Verstappen vs Norris 2024 (WINDOW + JOIN)")
print("Does form dip when conditions change? The rolling window smooths race noise.\n")
_ = run(q12)


## Query Coverage Summary

| Requirement | Queries |
|---|---|
| **10+ SQL queries** | 12 queries above |
| **3+ JOINs** | Q3 (results+qualifying), Q4 (three-way), Q5 (results+weather), Q10 (self-join), Q12 (results+weather) |
| **2+ Window Functions** | Q6 (SUM OVER), Q7 (RANK OVER), Q12 (AVG OVER with ROWS BETWEEN) |
| **2+ GROUP BY / ROLLUP / CUBE** | Q1, Q2, Q3, Q4, Q5, Q8, Q9, Q10, Q11 all use GROUP BY |
| **2+ Subqueries** | Q8 (correlated subquery in HAVING), Q9 (scalar subquery in HAVING), Q6 (subquery in WHERE for top-6 filter) |

All queries are relevant to the central analysis question: does driver skill or
car quality matter more, and how do adverse conditions shift that balance?
